# SST Indices Preprocessing

This notebook runs the generalized SST index preprocessing workflow. It calls `scripts/run_process_sst_index.py` to:
1. Load monthly SST inputs for E3SM, observations (HadISST2), and CESM-SMYLE.
2. Compute monthly and seasonal SST indices, including Nino regions, TNA, TSA, IOD, TNI, ONI, RONI, and Atlantic indices.
3. Compute ELI from full SST fields for E3SM MPAS-Ocean history files and CESM-SMYLE monthly TS benchmark files.
4. Save SST-index time-series outputs into the clean, model-first `sst_index/timeseries` layout used by the downstream diagnostics.

Outputs are written under:
- `JRA55_FOSIRL/sst_index/timeseries` and `Reanalysis/sst_index/timeseries` for E3SM cases.
- `HadISST2/sst_index/timeseries` for observations.
- `CESM-SMYLE/sst_index/timeseries` for the CESM-SMYLE benchmark.

The ELI section is included here because ELI is part of the same SST-index data product, even though its computation uses full-field centroid logic instead of a rectangular regional mean.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

# Identify repository root
REPO_ROOT = Path(os.getcwd())
if not (REPO_ROOT / "scripts").exists():
    REPO_ROOT = Path("..")

In [2]:
# -----------------------------
# Configuration
# -----------------------------
# Multi-case E3SM hindcasts. Add more entries here as new post-processed
# hindcasts land in different data directories.
E3SM_CASES = {
    "E3SM-FOSIRL": {
        "data_dir": "/global/cfs/cdirs/e3sm/S2S2D/post_process",
        "eli_data_dir": "/global/cfs/cdirs/e3smdata/simulations/S2S2D",
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
        "cache_tag": "JRA55_FOSIRL",
        "display_name": "E3SMv3-FOSIRL",
    },
    "E3SM-Reanalysis": {
        "data_dir": "/global/cfs/cdirs/e3sm/S2S2D/post_process",
        "eli_data_dir": "/global/cfs/cdirs/e3sm/S2S2D/simulation",
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce",
        "cache_tag": "Reanalysis",
        "display_name": "E3SMv3-Reanalysis",
    },
#    "E3SM-4DEnVarOcn": {
#        "data_dir": "/global/cfs/cdirs/e3sm/S2S2D/post_process",
#        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn",
#        "cache_tag": "4DEnVarOcn",
#        "display_name": "E3SMv3-4DEnVarOcn",
#    },
}

CONFIG = {
    "sources": ["obs", "e3sm", "smyle"],
    "e3sm_cases": E3SM_CASES,
    "regions": [
        "IOD", "TNI", "ONI", "RONI",
        "Nino12", "Nino3", "Nino3.4", "Nino4",
        "TNA", "TSA", "PACWRAMPOOL", "AtlNino", "AtlMDR"
    ],  # List of target indices/regions to compute
    "custom_regions": {
        "Nino12": {
            "lonlat": [270.0, 280.0, -10.0, 0.0],
            "long_name": "Nino 1+2 regional mean SST",
        },
        "Nino3": {
            "lonlat": [210.0, 270.0, -5.0, 5.0],
            "long_name": "Nino 3 regional mean SST",
        },
        "Nino3.4": {
            "lonlat": [190.0, 240.0, -5.0, 5.0],
            "long_name": "Nino 3.4 regional mean SST",
        },
        "Nino4": {
            "lonlat": [160.0, 210.0, -5.0, 5.0],
            "long_name": "Nino 4 regional mean SST",
        },
        "TNA": {
            "lonlat": [305.0, 345.0, 5.0, 25.0],
            "long_name": "TNA regional mean SST",
        },
        "TSA": {
            "lonlat": [330.0, 10.0, -20.0, 0.0],
            "long_name": "TSA regional mean SST",
        },
        "PACWRAMPOOL": {
            "lonlat": [60.0, 170.0, -15.0, 15.0],
            "long_name": "PACWRAMPOOL regional mean SST",
        },
        "AtlNino": {
            "lonlat": [340.0, 360.0, -3.0, 3.0],
            "long_name": "Atlantic Nino regional mean SST",
        },
        "AtlMDR": {
            "lonlat": [280.0, 350.0, 10.0, 20.0],
            "long_name": "Atlantic MDR regional mean SST",
        },
        # Helper regions for derived indices
        "IOD_West": {
            "lonlat": [50.0, 70.0, -10.0, 10.0],
            "long_name": "IOD West regional mean SST",
        },
        "IOD_East": {
            "lonlat": [90.0, 110.0, -10.0, 0.0],
            "long_name": "IOD East regional mean SST",
        },
        "TropicalMean": {
            "lonlat": [0.0, 360.0, -20.0, 20.0],
            "long_name": "Tropical Mean regional mean SST",
        },
    },
    "outdir": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag",
    "smyle_outdir": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE",
    "obs_outdir": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/HadISST2/sst_index/timeseries",
    "init_months": [5, 11],
    "year_start": 1980,
    "year_end": 2018,
    "climy0": 1980,
    "climy1": 2010,
    "nlead": 24,
    "e3sm_nens": 10,
    "smyle_nens": 20,
    "workers": 8,
    "force": True,  # Set to True to force rewrite
    "process_eli": True,
    # "native" uses MPAS-Ocean history files; "regridded" uses the same
    # regridded TS inputs as the other SST indices.
    "eli_input_grid": "native",
    # Generate missing CESM-SMYLE TS monthly/seasonal benchmark inputs before
    # computing CESM-SMYLE SST indices. Existing files are not overwritten.
    "auto_generate_smyle_benchmark": True,
}


In [ ]:
# -----------------------------
# Construct and execute command for each index/case
# -----------------------------
import json

script_path = str(REPO_ROOT / "scripts" / "run_process_sst_index.py")

# Set environment variables for GDAL/PROJ
env = os.environ.copy()
conda_prefix = "/global/homes/z/zhan391/.conda/envs/e3sm_analysis"
env["GDAL_DATA"] = f"{conda_prefix}/share/gdal"
env["PROJ_LIB"] = f"{conda_prefix}/share/proj"


def build_base_cmd(sources, region):
    cmd = [
        sys.executable,
        script_path,
        "--sources", *sources,
        "--outdir", CONFIG["outdir"],
        "--smyle-outdir", CONFIG["smyle_outdir"],
        "--obs-outdir", CONFIG["obs_outdir"],
        "--init-months", *(str(m) for m in CONFIG["init_months"]),
        "--year-start", str(CONFIG["year_start"]),
        "--year-end", str(CONFIG["year_end"]),
        "--climy0", str(CONFIG["climy0"]),
        "--climy1", str(CONFIG["climy1"]),
        "--nlead", str(CONFIG["nlead"]),
        "--e3sm-nens", str(CONFIG["e3sm_nens"]),
        "--smyle-nens", str(CONFIG["smyle_nens"]),
        "--workers", str(CONFIG["workers"]),
        "--regions", region,
    ]

    if CONFIG.get("custom_regions"):
        cmd.extend(["--custom-regions", json.dumps(CONFIG["custom_regions"])])

    if CONFIG["force"]:
        cmd.append("--force")

    return cmd


def run_cmd(cmd, label, region):
    print("=" * 60)
    print(f"Processing SST index: {region} | {label}")
    print("=" * 60)
    print("Running command:")
    print(" ".join(cmd))

    result = subprocess.run(cmd, env=env, capture_output=True, text=True)

    print("\n--- STDOUT ---")
    print(result.stdout)

    if result.returncode != 0:
        print("\n--- STDERR ---")
        print(result.stderr)
        raise RuntimeError(
            f"Preprocessing failed for '{label}' index '{region}' "
            f"with exit code {result.returncode}"
        )




def smyle_benchmark_files():
    root = Path(CONFIG["smyle_outdir"]) / "leadtime_acc" / "inputs" / "TS"
    files = []
    for init_month in CONFIG["init_months"]:
        for freq in ("mon", "seas"):
            files.append(
                root / f"BSMYLE{init_month:02d}_TS_N{CONFIG['smyle_nens']:02d}_M{CONFIG['nlead']:02d}_{freq}.nc"
            )
    return files


def build_smyle_benchmark_cmd():
    return [
        sys.executable,
        str(REPO_ROOT / "scripts" / "run_process_cesm_smyle_benchmark.py"),
        "--fields", "TS",
        "--init-months", *(str(m) for m in CONFIG["init_months"]),
        "--year-start", str(CONFIG["year_start"]),
        "--year-end", str(CONFIG["year_end"]),
        "--nens", str(CONFIG["smyle_nens"]),
        "--nlead", str(CONFIG["nlead"]),
        "--workers", str(CONFIG["workers"]),
        "--freqs", "mon", "seas",
        "--outdir", CONFIG["smyle_outdir"],
    ]


def ensure_smyle_benchmark_inputs():
    if "smyle" not in CONFIG["sources"]:
        return

    missing = [path for path in smyle_benchmark_files() if not path.exists()]
    if not missing:
        print("CESM-SMYLE TS benchmark inputs already exist.")
        return

    cmd = build_smyle_benchmark_cmd()
    if not CONFIG.get("auto_generate_smyle_benchmark", False):
        missing_text = "\n".join(f"  {path}" for path in missing)
        raise FileNotFoundError(
            "Missing CESM-SMYLE TS benchmark inputs needed for SST indices:\n"
            f"{missing_text}\n\n"
            "Generate them with:\n"
            + " ".join(cmd)
        )

    print("Missing CESM-SMYLE TS benchmark inputs; generating them now:")
    print(" ".join(cmd))
    result = subprocess.run(cmd, env=env, capture_output=True, text=True)
    print("\n--- SMYLE BENCHMARK STDOUT ---")
    print(result.stdout)
    if result.returncode != 0:
        print("\n--- SMYLE BENCHMARK STDERR ---")
        print(result.stderr)
        raise RuntimeError(
            f"CESM-SMYLE TS benchmark generation failed with exit code {result.returncode}"
        )

    still_missing = [path for path in smyle_benchmark_files() if not path.exists()]
    if still_missing:
        missing_text = "\n".join(f"  {path}" for path in still_missing)
        raise FileNotFoundError(
            "CESM-SMYLE benchmark generation completed, but these expected files are still missing:\n"
            f"{missing_text}"
        )


def sst_index_regions_to_process():
    regions = list(CONFIG["regions"])
    if CONFIG.get("process_eli", True) and "ELI" not in regions:
        regions.append("ELI")
    return regions


def should_process_e3sm_region(region):
    if region == "ELI" and CONFIG.get("eli_input_grid", "native") != "regridded":
        return False
    return True


ensure_smyle_benchmark_inputs()

shared_sources = [s for s in CONFIG["sources"] if s != "e3sm"]
process_e3sm = "e3sm" in CONFIG["sources"]

for r in sst_index_regions_to_process():
    if shared_sources:
        cmd = build_base_cmd(shared_sources, r)
        run_cmd(cmd, "+".join(shared_sources), r)

    if process_e3sm and should_process_e3sm_region(r):
        for case_key, case_info in CONFIG["e3sm_cases"].items():
            cmd = build_base_cmd(["e3sm"], r)
            cmd.extend([
                "--e3sm-data-dir", case_info["data_dir"],
                "--e3sm-case-prefix", case_info["case_prefix"],
                "--e3sm-cache-tag", case_info["cache_tag"],
                "--e3sm-display-name", case_info.get("display_name", case_key),
            ])
            run_cmd(cmd, case_key, r)

print("\nAll SST indices processed successfully!")


## ELI Index Preprocessing

ELI is part of the SST-index product, but it is computed from full SST fields rather than rectangular regional means. This section writes E3SM and CESM-SMYLE ELI files into the same `sst_index/timeseries` output directories used above.

In [4]:
# ELI imports and native-library paths.
_env_prefix = sys.prefix
_proj_path = os.path.join(_env_prefix, "share", "proj")
if os.path.isfile(os.path.join(_proj_path, "proj.db")):
    os.environ["CONDA_PREFIX"] = _env_prefix
    os.environ["PROJ_LIB"] = _proj_path
    os.environ["PROJ_DATA"] = _proj_path

import warnings
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import xarray as xr

print(f"Python      : {sys.executable}")
print(f"CONDA_PREFIX: {os.environ.get('CONDA_PREFIX', '(not set)')}")


Python      : /global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python
CONDA_PREFIX: /global/homes/z/zhan391/.conda/envs/e3sm_analysis


In [ ]:
# ------------------------------------------------------------------ #
#  ELI configuration
# ------------------------------------------------------------------ #

ELI_INPUT_GRID = CONFIG.get("eli_input_grid", "native")
PROCESS_ELI = CONFIG.get("process_eli", True) and ELI_INPUT_GRID == "native"
S2D_DIAG_ROOT = Path(CONFIG["outdir"])

# MPAS-Ocean mesh file for E3SM native-ocean ELI.
MESH_FILE = Path(
    "/global/cfs/cdirs/e3sm/inputdata/ocn/mpas-o/IcoswISC30E3r5/"
    "mpaso.IcoswISC30E3r5.rstFromG-chrysalis.20231121.nc"
)

# ELI uses native MPAS-Ocean history files, not the post-processed atm files.
E3SM_CASES = {}
if "e3sm" in CONFIG["sources"]:
    for case_key, case_info in CONFIG["e3sm_cases"].items():
        E3SM_CASES[case_key] = {
            **case_info,
            "data_dir": case_info.get("eli_data_dir", case_info["data_dir"]),
        }

START_YEAR = int(CONFIG["year_start"])
END_YEAR = int(CONFIG["year_end"])
EXCL_YEAR = None
INIT_MONTHS = list(CONFIG["init_months"])
lead_years = [y for y in np.arange(START_YEAR, END_YEAR + 1) if y != EXCL_YEAR]

CASES_BY_E3SM_CASE = {}
CASE_TO_DATA_DIR = {}
OUTDIR_BY_E3SM_CASE = {}
for case_key, case_info in E3SM_CASES.items():
    data_dir = Path(case_info["data_dir"])
    case_prefix = case_info["case_prefix"]
    cache_tag = case_info["cache_tag"]
    cases = [
        f"{case_prefix}_{year}{init_month:02d}0100"
        for init_month in INIT_MONTHS
        for year in lead_years
    ]
    CASES_BY_E3SM_CASE[case_key] = cases
    OUTDIR_BY_E3SM_CASE[case_key] = S2D_DIAG_ROOT / cache_tag / "sst_index" / "timeseries"
    for case in cases:
        CASE_TO_DATA_DIR[case] = data_dir

CASES = [case for cases in CASES_BY_E3SM_CASE.values() for case in cases]

# Non-E3SM sources do not have a native/regridded ELI split. They are
# processed by run_process_sst_index.py through the normal regridded path.
PROCESS_CESM_SMYLE = False
CESM_SMYLE_OUTDIR = Path(CONFIG["smyle_outdir"])
CESM_SMYLE_TS_DIR = CESM_SMYLE_OUTDIR / "leadtime_acc" / "inputs" / "TS"
CESM_SMYLE_ELI_OUTDIR = CESM_SMYLE_OUTDIR / "sst_index" / "timeseries"
CESM_SMYLE_NENS = int(CONFIG["smyle_nens"])

CASE_NENS = int(CONFIG["e3sm_nens"])
MEMBERS = [f"EN{i:02d}" for i in range(CASE_NENS)]
NENS = None

ELI_LAT_MIN = -5.0
ELI_LAT_MAX = 5.0
ELI_LON_MIN = 120.0
ELI_LON_MAX = 290.0
TC_LAT_HALF = 5.0

OCN_HIST_PATTERN = "*mpaso.hist.am.timeSeriesStatsMonthly.*.nc"
OCN_SST_VAR = "timeMonthly_avg_activeTracers_temperature"

NLEAD = int(CONFIG["nlead"])
FORCE_REWRITE = bool(CONFIG["force"])
NWORKERS = max(1, int(CONFIG["workers"]))

print(f"ELI_INPUT_GRID: {ELI_INPUT_GRID}")
print(f"PROCESS_ELI   : {PROCESS_ELI}")
print(f"NLEAD         : {NLEAD}")
print(f"CASE_NENS     : {CASE_NENS}")
print(f"FORCE_REWRITE : {FORCE_REWRITE}")
print(f"NWORKERS      : {NWORKERS}")
print("CESM-SMYLE/obs ELI: handled by the regular regridded SST-index command path")
for case_key, cases in CASES_BY_E3SM_CASE.items():
    case_info = E3SM_CASES[case_key]
    print(f"  {case_key}: {len(cases)} cases")
    print(f"    data_dir : {case_info['data_dir']}")
    print(f"    outdir   : {OUTDIR_BY_E3SM_CASE[case_key]}")


### Validate ELI Inputs

In [ ]:
if not PROCESS_ELI:
    print("PROCESS_ELI is False; skipping ELI validation.")
else:
    # ---- enforce all required fields are set ----
    _required = {
        "S2D_DIAG_ROOT":       S2D_DIAG_ROOT,
        "E3SM_CASES":          E3SM_CASES,
        "CASES_BY_E3SM_CASE":  CASES_BY_E3SM_CASE,
        "OUTDIR_BY_E3SM_CASE": OUTDIR_BY_E3SM_CASE,
        "MESH_FILE":           MESH_FILE,
        "CASES":               CASES,
        "MEMBERS":             MEMBERS,
    }
    _missing = [k for k, v in _required.items() if v is None]
    if _missing:
        raise ValueError(
            "The following required fields are still None - fill them in the "
            "Configuration cell:\n" + "\n".join(f"  {k}" for k in _missing)
        )

    # ---- basic configuration checks ----
    _errors = []
    if not E3SM_CASES:
        _errors.append("E3SM_CASES is empty.")
    if not CASES:
        _errors.append("CASES is empty; check START_YEAR, END_YEAR, EXCL_YEAR, and INIT_MONTHS.")
    if not lead_years:
        _errors.append("lead_years is empty; check START_YEAR, END_YEAR, and EXCL_YEAR.")
    if NLEAD <= 0:
        _errors.append(f"NLEAD must be positive, got {NLEAD}.")
    if CASE_NENS <= 0:
        _errors.append(f"CASE_NENS must be positive, got {CASE_NENS}.")
    if NENS is not None and NENS <= 0:
        _errors.append(f"NENS must be positive or None, got {NENS}.")
    if NWORKERS <= 0:
        _errors.append(f"NWORKERS must be positive, got {NWORKERS}.")
    if not MESH_FILE.is_file():
        _errors.append(f"MESH_FILE does not exist: {MESH_FILE}")

    if PROCESS_CESM_SMYLE:
        if not CESM_SMYLE_TS_DIR.is_dir():
            _errors.append(f"CESM-SMYLE monthly TS benchmark directory not found: {CESM_SMYLE_TS_DIR}")
        else:
            for init_month in INIT_MONTHS:
                _smyle_file = CESM_SMYLE_TS_DIR / f"BSMYLE{init_month:02d}_TS_N{CESM_SMYLE_NENS:02d}_M{NLEAD:02d}_mon.nc"
                if not _smyle_file.is_file():
                    _errors.append(
                        "CESM-SMYLE monthly TS benchmark file not found: "
                        f"{_smyle_file}. Run 0_run_cesm_smyle_benchmark.ipynb first."
                    )

    # ---- check paths and reference members for each E3SM case group ----
    members_by_case = {}
    for case_key, case_info in E3SM_CASES.items():
        data_dir = Path(case_info["data_dir"])
        case_names = CASES_BY_E3SM_CASE.get(case_key, [])
        if not data_dir.is_dir():
            _errors.append(f"data_dir does not exist for {case_key}: {data_dir}")
            continue
        if not case_names:
            _errors.append(f"No cases generated for {case_key}.")
            continue

        ref_case_dir = data_dir / case_names[0]
        if not ref_case_dir.is_dir():
            _errors.append(f"First case directory does not exist for {case_key}: {ref_case_dir}")
            continue

        all_members = sorted(p.name for p in ref_case_dir.iterdir()
                             if p.is_dir() and p.name.startswith("EN"))
        if not all_members:
            _errors.append(f"No EN* member directories found in {ref_case_dir}")
            continue

        if MEMBERS:
            missing_members = sorted(set(MEMBERS) - set(all_members))
            if missing_members:
                _errors.append(
                    "Requested member directories are missing in the reference case "
                    f"{ref_case_dir}: {missing_members}"
                )
                continue
            selected = list(MEMBERS)
        else:
            selected = all_members

        if NENS is not None:
            if NENS > len(selected):
                _errors.append(f"NENS={NENS} exceeds available members for {case_key} ({len(selected)}).")
                continue
            selected = selected[:NENS]

        if not selected:
            _errors.append(f"No members selected for {case_key}.")
            continue
        members_by_case[case_key] = selected

    if _errors:
        raise ValueError("Configuration validation failed:\n" + "\n".join(_errors))

    if PROCESS_CESM_SMYLE:
        CESM_SMYLE_ELI_OUTDIR.mkdir(parents=True, exist_ok=True)
        if not os.access(CESM_SMYLE_ELI_OUTDIR, os.W_OK):
            raise PermissionError(f"CESM-SMYLE output directory is not writable: {CESM_SMYLE_ELI_OUTDIR}")

    for outdir in OUTDIR_BY_E3SM_CASE.values():
        outdir.mkdir(parents=True, exist_ok=True)
        if not os.access(outdir, os.W_OK):
            raise PermissionError(f"Output directory is not writable: {outdir}")

    # Use the requested/available member list from the first configured group as the
    # output ensemble convention. All enabled groups should use the same member set.
    _first_case_key = next(iter(members_by_case))
    members_avail = members_by_case[_first_case_key]
    RUN_NENS = len(members_avail)
    OUT_NENS = RUN_NENS

    for case_key, selected in members_by_case.items():
        if selected != members_avail:
            raise ValueError(
                f"Member selection differs for {case_key}: {selected}; expected {members_avail}."
            )

    print("Configuration valid")
    print(f"  S2D_DIAG_ROOT : {S2D_DIAG_ROOT}")
    print(f"  MESH_FILE     : {MESH_FILE}")
    for case_key, cases in CASES_BY_E3SM_CASE.items():
        print(f"  {case_key}:")
        print(f"    data_dir : {E3SM_CASES[case_key]['data_dir']}")
        print(f"    outdir   : {OUTDIR_BY_E3SM_CASE[case_key]}")
        print(f"    cases    : {len(cases)} total  ({cases[0]}  ...  {cases[-1]})")
    print(f"  Members       : {members_avail}  (RUN_NENS={RUN_NENS}, NENS={NENS})")
    if PROCESS_CESM_SMYLE:
        print(f"  CESM-SMYLE input : {CESM_SMYLE_TS_DIR}")
        print(f"  CESM-SMYLE outdir: {CESM_SMYLE_ELI_OUTDIR}")


### Load MPAS-Ocean Mesh Geometry

In [7]:
if not PROCESS_ELI:
    print("PROCESS_ELI is False; skipping ELI mesh setup.")
else:
    with xr.open_dataset(MESH_FILE) as mesh:
        for mesh_var in ["latCell", "lonCell", "areaCell"]:
            if mesh_var not in mesh:
                raise KeyError(f"Required mesh variable {mesh_var!r} is missing from {MESH_FILE}")

        lat  = mesh["latCell"].values  * 180.0 / np.pi   # degrees North
        lon  = mesh["lonCell"].values  * 180.0 / np.pi   # degrees East  [0, 360]
        area = mesh["areaCell"].values                    # m²

    lon = np.mod(lon, 360.0)

    if not (np.isfinite(lat).all() and np.isfinite(lon).all()):
        raise ValueError("Mesh latitude/longitude contains non-finite values.")
    if not (np.isfinite(area).all() and np.all(area > 0.0)):
        raise ValueError("Mesh cell areas must be finite and strictly positive.")

    # Equatorial Pacific region for ELI centroid.
    region_eq = (
        (lat >= ELI_LAT_MIN) & (lat <= ELI_LAT_MAX) &
        (lon >= ELI_LON_MIN) & (lon <= ELI_LON_MAX)
    )

    # Broad tropical band for reference temperature Tc.
    region_tropics = (lat >= -TC_LAT_HALF) & (lat <= TC_LAT_HALF)

    if not np.any(region_eq):
        raise ValueError("Equatorial Pacific mask selected zero mesh cells; check ELI bounds.")
    if not np.any(region_tropics):
        raise ValueError("Tropical reference mask selected zero mesh cells; check TC_LAT_HALF.")

    # Integer indices are faster and cleaner for repeated file reads.
    idx_union = np.flatnonzero(region_eq | region_tropics)
    eq_on_union = region_eq[idx_union]
    tropics_on_union = region_tropics[idx_union]

    lon_eq  = lon[idx_union][eq_on_union].astype(np.float32)
    area_eq = area[idx_union][eq_on_union].astype(np.float64)
    area_tr = area[idx_union][tropics_on_union].astype(np.float64)

    print(f"Mesh cells total    : {len(lat):,}")
    print(f"Equatorial Pacific  : {region_eq.sum():,} cells  "
          f"(lat [{ELI_LAT_MIN}, {ELI_LAT_MAX}]°, lon [{ELI_LON_MIN}, {ELI_LON_MAX}]°)")
    print(f"Tropical band       : {region_tropics.sum():,} cells  (lat ±{TC_LAT_HALF}°)")

Warning 3: Cannot find header.dxf (GDAL_DATA is not defined)


Mesh cells total    : 465,044
Equatorial Pacific  : 24,009 cells  (lat [-5.0, 5.0]°, lon [120.0, 290.0]°)
Tropical band       : 42,941 cells  (lat ±5.0°)


### Compute E3SM ELI

In [ ]:
if not PROCESS_ELI:
    print("PROCESS_ELI is False; skipping E3SM ELI.")
else:
    def _weighted_mean_skipna_1d(values: np.ndarray, weights: np.ndarray) -> float:
        """Area-weighted mean for one time slice, ignoring non-finite values."""
        finite = np.isfinite(values)
        if not np.any(finite):
            return np.nan
        denom = np.sum(weights[finite])
        if denom <= 0.0:
            return np.nan
        return float(np.sum(values[finite] * weights[finite]) / denom)


    def _read_surface_sst_on_cells(path: Path, idx_union: np.ndarray) -> np.ndarray:
        """Read one monthly MPAS-Ocean file on the needed cells only."""
        with xr.open_dataset(path, decode_times=False) as ds:
            if OCN_SST_VAR not in ds:
                raise KeyError(f"{OCN_SST_VAR!r} not found in {path}")

            sst_da = ds[OCN_SST_VAR]
            if "nVertLevels" in sst_da.dims:
                sst_da = sst_da.isel(nVertLevels=0)
            if "Time" in sst_da.dims:
                sst_da = sst_da.isel(Time=0)
            if "nCells" not in sst_da.dims:
                raise ValueError(f"{OCN_SST_VAR!r} must have nCells dimension; found {sst_da.dims}")

            sst = sst_da.isel(nCells=idx_union).astype("float32").load().values

        if sst.ndim != 1:
            raise ValueError(f"Expected 1-D SST after slicing {path}, got shape {sst.shape}")
        return sst


    def compute_eli_member(
        hist_dir: Path,
        pattern: str,
        idx_union: np.ndarray,
        eq_on_union: np.ndarray,
        tropics_on_union: np.ndarray,
        lon_eq: np.ndarray,
        area_eq: np.ndarray,
        area_tr: np.ndarray,
        nlead: int,
    ) -> np.ndarray:
        """
        Compute monthly ELI for one ensemble member.

        This reads files one-by-one instead of using ``open_mfdataset``. For this
        workflow that is usually faster because each member has a small fixed number
        of monthly files and only one variable/cell subset is needed.
        """
        files = sorted(hist_dir.glob(pattern))
        if not files:
            return np.full(nlead, np.nan, dtype=np.float32)
        if len(files) < nlead:
            print(f"  [WARN] {hist_dir}: only {len(files)} monthly files found; expected {nlead}.")

        out = np.full(nlead, np.nan, dtype=np.float32)
        for lead_idx, path in enumerate(files[:nlead]):
            sst = _read_surface_sst_on_cells(path, idx_union)
            sst_eq = sst[eq_on_union]
            sst_tr = sst[tropics_on_union]

            Tc = _weighted_mean_skipna_1d(sst_tr, area_tr)
            if not np.isfinite(Tc):
                continue

            warm = np.isfinite(sst_eq) & (sst_eq > Tc)
            den = np.sum(area_eq[warm])
            if den > 0.0:
                out[lead_idx] = np.sum(area_eq[warm] * lon_eq[warm]) / den

        return out


    def _compute_case_member(args):
        yi, mi, case, member, hist_dir = args
        try:
            values = compute_eli_member(
                hist_dir, OCN_HIST_PATTERN,
                idx_union, eq_on_union, tropics_on_union,
                lon_eq, area_eq, area_tr,
                NLEAD,
            )
            return yi, mi, case, member, values, None
        except Exception as exc:
            return yi, mi, case, member, None, exc


    for case_key, case_info in E3SM_CASES.items():
        data_dir = Path(case_info["data_dir"])
        case_prefix = case_info["case_prefix"]
        cache_tag = case_info["cache_tag"]
        display_name = case_info.get("display_name", case_key)
        outdir = OUTDIR_BY_E3SM_CASE[case_key]

        print(f"\n############################################################")
        print(f"# E3SM case group: {case_key} ({display_name})")
        print(f"# data_dir: {data_dir}")
        print(f"# outdir  : {outdir}")
        print(f"############################################################")

        for init_month in INIT_MONTHS:
            outfile = outdir / f"E3SMLE{init_month:02d}_ELI_native_N{OUT_NENS:02d}_M{NLEAD:02d}.nc"

            if outfile.exists() and not FORCE_REWRITE:
                print(f"Skipping {case_key} init month {init_month:02d} - output exists: {outfile}")
                continue

            print(f"\n=== {case_key} init month {init_month:02d} ===")
            t0 = time.time()

            eli_all = np.full(
                (len(lead_years), NLEAD, RUN_NENS),
                np.nan, dtype=np.float32,
            )

            cases_for_month = [
                f"{case_prefix}_{year}{init_month:02d}0100" for year in lead_years
            ]

            workers = min(int(NWORKERS), RUN_NENS)

            for yi, (year, case) in enumerate(zip(lead_years, cases_for_month)):
                case_dir = data_dir / case
                if not case_dir.is_dir():
                    print(f"  [WARN] Case directory not found: {case_dir}")
                    continue

                tasks = [
                    (yi, mi, case, member, case_dir / member / "archive" / "ocn" / "hist")
                    for mi, member in enumerate(members_avail)
                ]

                if workers == 1:
                    results_iter = map(_compute_case_member, tasks)
                    pool = None
                else:
                    pool = ThreadPoolExecutor(max_workers=workers)
                    futures = [pool.submit(_compute_case_member, task) for task in tasks]
                    results_iter = (future.result() for future in as_completed(futures))

                try:
                    for _, mi, case_name, member, values, exc in results_iter:
                        if exc is not None:
                            print(f"  [ERROR] {case_name}/{member}: {exc}")
                            continue
                        eli_all[yi, :, mi] = values
                finally:
                    if pool is not None:
                        pool.shutdown(wait=True)

                n_ok = int(np.sum(np.isfinite(eli_all[yi])))
                print(f"  [{yi + 1:3d}/{len(lead_years)}] {case}  - {n_ok}/{NLEAD * RUN_NENS} valid values")

            # ---- build output dataset ----
            ds_out = xr.Dataset(
                {
                    "eli": xr.DataArray(
                        eli_all,
                        dims=("Y", "L", "M"),
                        coords={
                            "Y": np.array(lead_years, dtype=np.int32),
                            "L": np.arange(1, NLEAD + 1, dtype=np.int32),
                            "M": np.arange(RUN_NENS, dtype=np.int32),
                        },
                        attrs={
                            "long_name": "Equatorial Longitude Index",
                            "units": "degrees_east",
                            "description": (
                                "Area-weighted centroid longitude of warm SST cells "
                                "(SST > tropical-mean SST) in the equatorial Pacific "
                                f"(lat {ELI_LAT_MIN}-{ELI_LAT_MAX} deg, "
                                f"lon {ELI_LON_MIN}-{ELI_LON_MAX} deg).  "
                                f"Tc reference band: +/-{TC_LAT_HALF} deg."
                            ),
                        },
                    )
                }
            )

            ds_out["Y"].attrs = {"long_name": "initialization year", "units": "year"}
            ds_out["L"].attrs = {"long_name": "forecast lead month", "units": "months"}
            ds_out["M"].attrs = {"long_name": "ensemble member index"}

            ds_out["member_id"] = xr.DataArray(
                np.array(members_avail, dtype="U5"),
                dims="M",
                attrs={"long_name": "ensemble member label"},
            )

            ds_out.attrs = {
                "case_key":     case_key,
                "display_name": display_name,
                "cache_tag":    cache_tag,
                "case_prefix":  case_prefix,
                "data_dir":     str(data_dir),
                "init_month":   int(init_month),
                "start_year":   int(START_YEAR),
                "end_year":     int(END_YEAR),
                "case_nens":    int(CASE_NENS),
                "run_nens":     int(RUN_NENS),
                "nworkers":     int(workers),
                "eli_lat_min":  ELI_LAT_MIN,
                "eli_lat_max":  ELI_LAT_MAX,
                "eli_lon_min":  ELI_LON_MIN,
                "eli_lon_max":  ELI_LON_MAX,
                "tc_lat_half":  TC_LAT_HALF,
            }

            # Atomic write
            tmp_file = outfile.with_suffix(".tmp.nc")
            try:
                ds_out.to_netcdf(
                    tmp_file,
                    encoding={"eli": {"zlib": True, "complevel": 1, "dtype": "float32"}},
                )
                os.replace(tmp_file, outfile)
            finally:
                if tmp_file.exists():
                    tmp_file.unlink()

            elapsed = time.time() - t0
            n_nan = int(np.isnan(eli_all).sum())
            n_tot = eli_all.size
            print(f"  Saved -> {outfile}  ({elapsed:.0f}s, {n_nan}/{n_tot} NaN values)")


### Compute CESM-SMYLE ELI

In [ ]:
if not PROCESS_ELI:
    print("PROCESS_ELI is False; skipping CESM-SMYLE ELI.")
else:
    def _latlon_region_mask(lat_coord, lon_coord, lat_min, lat_max, lon_min, lon_max):
        """Return a 2D boolean mask on rectilinear lat/lon coordinates."""
        lat_da = xr.DataArray(lat_coord.values, dims=(lat_coord.dims[0],), coords={lat_coord.dims[0]: lat_coord.values})
        lon_vals = np.mod(lon_coord.values, 360.0)
        lon_da = xr.DataArray(lon_vals, dims=(lon_coord.dims[0],), coords={lon_coord.dims[0]: lon_coord.values})
        lon2d, lat2d = xr.broadcast(lon_da, lat_da)
        mask = (lat2d >= lat_min) & (lat2d <= lat_max) & (lon2d >= lon_min) & (lon2d <= lon_max)
        return mask.transpose(lat_coord.dims[0], lon_coord.dims[0])


    def compute_smyle_eli_from_ts(ts: xr.DataArray) -> xr.DataArray:
        """Compute ELI from CESM-SMYLE monthly TS with dims Y, L, M, lat, lon."""
        required_dims = {"Y", "L", "M", "lat", "lon"}
        missing = required_dims - set(ts.dims)
        if missing:
            raise ValueError(f"CESM-SMYLE TS is missing required dims: {sorted(missing)}")

        ts = ts.transpose("Y", "L", "M", "lat", "lon")
        lat_coord = ts["lat"]
        lon_coord = ts["lon"]

        eq_mask = _latlon_region_mask(lat_coord, lon_coord, ELI_LAT_MIN, ELI_LAT_MAX, ELI_LON_MIN, ELI_LON_MAX)
        tropics_mask = _latlon_region_mask(lat_coord, lon_coord, -TC_LAT_HALF, TC_LAT_HALF, 0.0, 360.0)

        lon2d, lat2d = xr.broadcast(
            xr.DataArray(np.mod(lon_coord.values, 360.0), dims=("lon",), coords={"lon": lon_coord.values}),
            lat_coord,
        )
        lon2d = lon2d.transpose("lat", "lon")
        weights = xr.DataArray(
            np.cos(np.deg2rad(lat_coord.values)),
            dims=("lat",),
            coords={"lat": lat_coord.values},
        )
        weights2d, _ = xr.broadcast(weights, ts["lon"])
        weights2d = weights2d.transpose("lat", "lon")

        # Use cos(lat) area weights. This is enough for the regular f09_g17 grid;
        # ELI only needs relative cell areas within latitude bands.
        tropics_weights = weights2d.where(tropics_mask, 0.0)
        tc = ts.weighted(tropics_weights).mean(("lat", "lon"), skipna=True)

        warm = ts > tc
        warm_weights = weights2d.where(eq_mask & warm, 0.0)
        den = warm_weights.sum(("lat", "lon"), skipna=True)
        num = (warm_weights * lon2d).sum(("lat", "lon"), skipna=True)
        eli = (num / den).astype("float32").rename("eli")
        eli = eli.where(den > 0)
        eli.attrs.update({
            "long_name": "Equatorial Longitude Index",
            "units": "degrees_east",
            "description": (
                "Area-weighted centroid longitude of warm TS grid cells "
                "(TS > tropical-mean TS) in the equatorial Pacific on CESM-SMYLE f09_g17."
            ),
        })
        return eli


    if PROCESS_CESM_SMYLE:
        CESM_SMYLE_ELI_OUTDIR.mkdir(parents=True, exist_ok=True)
        for init_month in INIT_MONTHS:
            infile = CESM_SMYLE_TS_DIR / f"BSMYLE{init_month:02d}_TS_N{CESM_SMYLE_NENS:02d}_M{NLEAD:02d}_mon.nc"
            outfile = CESM_SMYLE_ELI_OUTDIR / f"BSMYLE{init_month:02d}_ELI_N{CESM_SMYLE_NENS:02d}_M{NLEAD:02d}.nc"

            if outfile.exists() and not FORCE_REWRITE:
                print(f"Skipping CESM-SMYLE init month {init_month:02d} - output exists: {outfile}")
                continue

            print(f"\n=== CESM-SMYLE init month {init_month:02d} ===")
            print(f"  input : {infile}")
            print(f"  output: {outfile}")
            t0 = time.time()

            with xr.open_dataset(infile, chunks={"Y": 3, "L": -1, "M": 2, "lat": 96, "lon": 144}) as ds_smyle:
                if "TS" in ds_smyle:
                    ts = ds_smyle["TS"]
                elif "SST" in ds_smyle:
                    ts = ds_smyle["SST"]
                elif "sst" in ds_smyle:
                    ts = ds_smyle["sst"]
                else:
                    raise KeyError(f"No TS/SST/sst variable found in {infile}")

                eli = compute_smyle_eli_from_ts(ts).compute()
                ds_out = eli.to_dataset()
                ds_out["Y"].attrs = {"long_name": "initialization year", "units": "year"}
                ds_out["L"].attrs = {"long_name": "forecast lead month", "units": "months"}
                ds_out["M"].attrs = {"long_name": "ensemble member index"}
                ds_out.attrs.update({
                    "source": "CESM-SMYLE monthly TS benchmark",
                    "source_file": str(infile),
                    "cache_tag": "CESM-SMYLE",
                    "init_month": int(init_month),
                    "start_year": int(START_YEAR),
                    "end_year": int(END_YEAR),
                    "run_nens": int(CESM_SMYLE_NENS),
                    "eli_lat_min": ELI_LAT_MIN,
                    "eli_lat_max": ELI_LAT_MAX,
                    "eli_lon_min": ELI_LON_MIN,
                    "eli_lon_max": ELI_LON_MAX,
                    "tc_lat_half": TC_LAT_HALF,
                    "area_weights": "cos(latitude)",
                })

            tmp_file = outfile.with_suffix(".tmp.nc")
            try:
                ds_out.to_netcdf(
                    tmp_file,
                    encoding={"eli": {"zlib": True, "complevel": 1, "dtype": "float32"}},
                )
                os.replace(tmp_file, outfile)
            finally:
                if tmp_file.exists():
                    tmp_file.unlink()

            elapsed = time.time() - t0
            n_nan = int(np.isnan(ds_out["eli"].values).sum())
            n_tot = int(ds_out["eli"].size)
            print(f"  Saved -> {outfile}  ({elapsed:.0f}s, {n_nan}/{n_tot} NaN values)")
    else:
        print("PROCESS_CESM_SMYLE is False; skipping CESM-SMYLE ELI.")


### Verify ELI Outputs

In [ ]:
if not PROCESS_ELI:
    print("PROCESS_ELI is False; skipping ELI verification.")
else:
    import matplotlib.pyplot as plt

    for case_key, outdir in OUTDIR_BY_E3SM_CASE.items():
        print(f"\n=== {case_key}: {outdir} ===")
        for init_month in INIT_MONTHS:
            outfile = outdir / f"E3SMLE{init_month:02d}_ELI_native_N{OUT_NENS:02d}_M{NLEAD:02d}.nc"
            if not outfile.exists():
                print(f"[MISSING] {outfile}")
                continue

            with xr.open_dataset(outfile) as ds:
                eli = ds["eli"]
                n_valid = int(np.isfinite(eli).sum())
                n_total = int(eli.size)
                print(f"\nInit month {init_month:02d}  ->  {outfile.name}")
                print(f"  dims   : {dict(eli.sizes)}")
                if n_valid:
                    print(f"  min    : {float(eli.min(skipna=True)):.2f} degE")
                    print(f"  max    : {float(eli.max(skipna=True)):.2f} degE")
                    print(f"  mean   : {float(eli.mean(skipna=True)):.2f} degE")
                else:
                    print("  min/max/mean: unavailable (all values are NaN)")
                print(f"  valid  : {n_valid}/{n_total}")

                # Quick ensemble-mean time series for the first init year
                eli_yr0 = eli.isel(Y=0).values  # (L, M)
                fig, ax = plt.subplots(figsize=(9, 3))
                ax.plot(np.arange(1, NLEAD + 1), eli_yr0, color="gray", linewidth=0.7, alpha=0.6)
                ax.plot(np.arange(1, NLEAD + 1), np.nanmean(eli_yr0, axis=-1),
                        color="k", linewidth=2, label="Ens mean")
                ax.set_xlabel("Lead month")
                ax.set_ylabel("ELI (degE)")
                ax.set_title(
                    f"{case_key} ELI - init month {init_month:02d}, year {int(ds['Y'].isel(Y=0))}"
                )
                ax.legend(fontsize=9)
                plt.tight_layout()
                plt.show()

    if PROCESS_CESM_SMYLE:
        print(f"\n=== CESM-SMYLE: {CESM_SMYLE_ELI_OUTDIR} ===")
        for init_month in INIT_MONTHS:
            outfile = CESM_SMYLE_ELI_OUTDIR / f"BSMYLE{init_month:02d}_ELI_N{CESM_SMYLE_NENS:02d}_M{NLEAD:02d}.nc"
            if not outfile.exists():
                print(f"[MISSING] {outfile}")
                continue

            with xr.open_dataset(outfile) as ds:
                eli = ds["eli"]
                n_valid = int(np.isfinite(eli).sum())
                n_total = int(eli.size)
                print(f"\nCESM-SMYLE init month {init_month:02d}  ->  {outfile.name}")
                print(f"  dims   : {dict(eli.sizes)}")
                if n_valid:
                    print(f"  min    : {float(eli.min(skipna=True)):.2f} degE")
                    print(f"  max    : {float(eli.max(skipna=True)):.2f} degE")
                    print(f"  mean   : {float(eli.mean(skipna=True)):.2f} degE")
                else:
                    print("  min/max/mean: unavailable (all values are NaN)")
                print(f"  valid  : {n_valid}/{n_total}")
